# Does the attention of AgentFormer name the neighbour that matters?

DAMO 699 capstone. AgentFormer attention faithfulness.

**The research question.** Does the attention weight that AgentFormer puts on
one neighbour name the neighbour that the forecast of the ego depends on?

Three hypotheses answer that question.

- **H1.** Removal of the most-attended edge moves the forecast more than
  removal of the least-attended edge.
- **H2.** Removal of the most-attended edge moves the forecast more than
  removal of the nearest edge, so attention beats proximity.
- **H3.** The context of a window explains the faithfulness index.

**How to read this notebook.** The notebook reads the tables and the figures
that `scripts/run_all.py` writes. It fits no model and it runs no test. Run
the pipeline first. Then run every cell of this notebook from the top. A cell
whose file is absent prints one note and moves on.

In [ ]:
"""Set the path, import pandas, and define the two readers."""

import sys
from pathlib import Path

# Find the repository root, so the notebook runs from any working directory.
ROOT = Path.cwd().resolve()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

import config

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

try:
    from IPython.display import Image, display
except ImportError:
    # The check harness of scripts/make_notebook.py runs the cells outside a
    # kernel, so IPython is absent there. print() stands in for display().
    Image = None

    def display(value):
        """Print one value. This is the fallback outside a notebook."""
        print(value)


TABLES = Path(config.TABLE_DIR)
FIGURES = Path(config.FIGURE_DIR)
PROCESSED = Path(config.PROCESSED_DIR)


def show_table(path, rows: int = 25):
    """Show one table. Print a note and return None when the file is absent.

    The reader accepts a csv file and a parquet file. `rows` caps the shown
    rows. The return value is the whole frame, never the capped view.
    """
    path = Path(path)
    if not path.exists():
        print(f"absent: {path.name}. Run the stage that writes it.")
        return None
    frame = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
    print(f"{path.name}: {len(frame):,} rows, {len(frame.columns)} columns")
    display(frame.head(int(rows)))
    return frame


def show_figure(name: str):
    """Show one png of outputs/figures. Print a note when it is absent."""
    path = Path(FIGURES) / str(name)
    if not path.exists():
        print(f"absent: {path.name}. Run the stage that draws it.")
        return None
    if Image is None:
        print(f"figure on disk: {path}")
        return path
    display(Image(filename=str(path)))
    return path


print(f"root      {ROOT}")
print(f"tables    {TABLES}")
print(f"figures   {FIGURES}")
print(f"config    {config.config_hash()}")

## 1. The frozen design

`outputs/tables/design.csv` holds two rows. The **frozen** row states the
design that `config.py` and `src/hypotheses/design.py` fix before the first
ablation: the window stride, alpha, the sided rule, the primary removal count,
the minimum effect and the power. The **realised** row measures the frozen
windows table itself.

The two rows must agree closely. A large gap means that the exploratory sweep
and the frozen build no longer describe the same sample.

In [ ]:
design = show_table(TABLES / "design.csv")

## 2. The data

`eda_summary.csv` counts the rows, the pedestrians and the windows of every
scene. `eda_density.csv` reports the neighbourhood of the ego. Both tables come
from the stride 1 windows table, because the exploratory sweep needs every
window start.

In [ ]:
summary = show_table(TABLES / "eda_summary.csv")
density = show_table(TABLES / "eda_density.csv")

## 3. The design effect sweep

Windows overlap, so two rows are not independent. `eda_overlap.csv` reports the
intraclass correlation, the mean cluster size, the design effect and the
effective sample size at every stride. `src/hypotheses/design.py` reads the row
of the frozen stride and turns it into the power statement of section 1.

The figure shows the same sweep. A larger stride costs windows and buys
independence.

In [ ]:
sweep = show_table(TABLES / "eda_overlap.csv", rows=40)
show_figure("eda_overlap.png")

## 4. Accuracy and calibration

`predictions.parquet` holds the displacement errors of every window.
`calibration.csv` holds the coverage and the probability integral transform per
scene.

Note the determinism limit. AgentFormer is deterministic at inference and the
K draws are K fixed DLow modes, not K samples from a posterior. The coverage
therefore describes how the K fixed modes sit around the truth. It is not a
probabilistic calibration claim.

In [ ]:
predictions = show_table(PROCESSED / "predictions.parquet", rows=5)
if predictions is not None:
    scene = predictions["window_id"].astype(str).str.rsplit("_", n=1).str[0]
    measures = [
        name
        for name in ("ade", "fde", "minade_20", "minfde_20")
        if name in predictions.columns
    ]
    display(predictions.groupby(scene)[measures].describe().T)

calibration = show_table(TABLES / "calibration.csv")

## 5. Attention against distance

H2 asks whether attention beats proximity. The question only has an answer on
the windows where the two orders disagree. `arm_overlap.csv` holds the Jaccard
overlap of the top attention edges and the top distance edges, per window.

An overlap near 1 says that attention repeats the proximity order. H2 is then
weak before any test runs.

In [ ]:
overlap = show_table(TABLES / "arm_overlap.csv", rows=10)
if overlap is not None and "jaccard" in overlap.columns:
    print(overlap["jaccard"].describe().to_string())

## 6. The ablation sanity checks

`ablation_sanity.csv` holds the two checks of `scripts/run_ablation.py`. The
zero check masks nothing under fixed draws and must give a shift of exactly 0.
The noise floor masks nothing under free draws and reports the shift that the
sampling alone produces. AgentFormer is deterministic, so that floor is 0 and
any shift above 0 is a real change.

`attrition.csv` holds the arm attrition of `src/hypotheses/data_checks.py`.
Every window must carry every arm.

In [ ]:
sanity = show_table(TABLES / "ablation_sanity.csv")
attrition = show_table(TABLES / "attrition.csv", rows=40)

## 7. The perturbation curves

One curve is one arm. The point at a step is the mean shift over the windows
that hold that step. The band is a pairs cluster bootstrap on the connected
component. The second panel puts `morf` and `weight_matched` on the removed
mass axis, because `weight_matched` matches mass and not count. The third row
holds one panel per scene.

The band of a figure uses fewer resamples than a reported interval. Read a band
as a picture, never as a result.

In [ ]:
show_figure("hyp_curves.png")

## 8. The faithfulness index

The index compares the shift of the most-attended edge with the floor and the
ceiling of every single edge of the same window. An index of 1 says that
attention picked the strongest edge. An index of 0 says that attention is no
better than chance.

`fi_attrition.csv` counts, per scene, the windows at the frozen edge floor and
the windows whose index is null. A window at the floor of
`config.MIN_EGO_EDGES` reaches only +1 or -1, so it carries no middle value.

In [ ]:
fi_attrition = show_table(TABLES / "fi_attrition.csv")
show_figure("hyp_faithfulness.png")

## 9. H1 and H2

`hypotheses_extra.csv` holds the shape report of the paired differences, the
location in metres with its cluster bootstrap interval, and the share of
windows where the attention arm and the distance arm pick the same first edge.
`src/hypotheses/choose_test.py` reads that shape and names the test.

The figure holds the histogram of D for H1 and for H2. A D of exactly 0 is a
structural tie, and the Pratt rule keeps it, so the title counts the ties.

In [ ]:
extra = show_table(TABLES / "hypotheses_extra.csv")
show_figure("hyp_paired.png")

## 10. H3, the context of the faithfulness index

`h3_coefficients.csv` holds both fits of `src/hypotheses/h3_context.py`. The
headline fit drops the windows at the frozen edge floor, because those windows
give an index of exactly +1 or -1 and inject a spike that belongs to the edge
count and not to the context. `h3_vif.csv` holds the collinearity check of the
four covariates.

The figure draws the coefficients of the headline fit with a cluster-robust
interval.

In [ ]:
coefficients = show_table(TABLES / "h3_coefficients.csv", rows=40)
vif = show_table(TABLES / "h3_vif.csv")
show_figure("hyp_h3_coefficients.png")

## 11. The sensitivity table

`validate.csv` holds the descriptive checks of `src/hypotheses/validate.py`:
one scene held out at a time, the removal sweep, another attention module,
another time collapse, the second mask policy, and the noise floor. No p value
of this table enters a family. The table describes the result, it does not test
it again.

The forest plot shows the location per scene for H1 and for H2. Every scene
must point the same way, or the claim belongs to one scene alone.

In [ ]:
validate = show_table(TABLES / "validate.csv", rows=60)
show_figure("hyp_loso.png")

## 12. The verdicts

`verdicts.csv` is the answer. Holm corrects the three primary rows h1, h2 and
h3. Benjamini-Hochberg corrects the supporting family. A row rejects when the
adjusted p value is at or below `config.ALPHA`.

`n_clusters` is the count of connected components, never the count of windows.
The component is the cluster unit of the study, and it is the honest number
beside a p value.

In [ ]:
verdicts = show_table(TABLES / "verdicts.csv")
if verdicts is not None:
    columns = [
        name
        for name in (
            "hypothesis",
            "role",
            "effect_name",
            "effect",
            "p_raw",
            "p_adj",
            "n_clusters",
            "verdict",
        )
        if name in verdicts.columns
    ]
    display(verdicts.loc[:, columns])